In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)

print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /home/tuptc/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /home/tuptc/big-data-postgraduate
Salidas: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD02


In [2]:
# 1. Preparación de columnas
# Creación de "codigo_departamento" como texto limpio
eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip()

# Conversión a valores numéricos usando pd.to_numeric con errors="coerce" a "anio", "area_ha" y "produccion_t"
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip().str.replace(",", ".", regex=False), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip().str.replace(",", ".", regex=False), errors="coerce")

# Verificación de tipos de datos de lo modificado
print("--- Tipos de datos creados ---")
print(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].dtypes)

# Conteo de valores no convertibles generados
print("\n--- Conteo de NaN´s por columna ---")
ausencias = pd.Series({
    "codigo_departamento_nan": eva["codigo_departamento"].eq("").sum(),
    "anio_nan": eva["anio"].isna().sum(),
    "area_ha_nan": eva["area_ha"].isna().sum(),
    "produccion_t_nan": eva["produccion_t"].isna().sum()
})
print(ausencias)

# Comprobación de dimensiones (según la guía, se debe mantener exactamente 166.732 registros)
print(f"\nTotal de registros en eva: {len(eva)}")

--- Tipos de datos creados ---
codigo_departamento     object
anio                     Int64
area_ha                float64
produccion_t           float64
dtype: object

--- Conteo de NaN´s por columna ---
codigo_departamento_nan    0
anio_nan                   0
area_ha_nan                0
produccion_t_nan           0
dtype: int64

Total de registros en eva: 166732


In [3]:
# 2. Selección con máscara y loc
# Asignación de Máscara booleana para las 4 condiciones
mask = (
    eva["codigo_departamento"].eq("15") & #Boyacá?
    eva["anio"].isin([2023, 2024]) & #2023-2024
    eva["area_ha"].gt(0) & # Area < 0?
    eva["produccion_t"].ge(0) # Producción <= 0?
)

# Creación de subconjunto consulta_loc (extraccion de tuplas que cumplen con lo estipulado en la mascara)
consulta_loc = eva.loc[mask].copy()

# Calculo de comparativas (Tuplas al comienzo y al final del filtrado)
total_original = len(eva)
total_seleccionado = len(consulta_loc)
diferencia = total_original - total_seleccionado

print(f"Registros originales: {total_original:,}")
print(f"Registros seleccionados (Boyacá 2023-2024 validos): {total_seleccionado:,}")
print(f"Diferencia (Descartados): {diferencia:,}")

Registros originales: 166,732
Registros seleccionados (Boyacá 2023-2024 validos): 4,924
Diferencia (Descartados): 161,808


In [4]:
#3. Consulta equivalente con .query

# Variable con el codigo de boyacá
depto_boyaca = "15"

# Replicarción de la máscara usando .query con engine="python"
consulta_query = eva.query(
    'codigo_departamento == @depto_boyaca and anio in [2023, 2024] and area_ha > 0 and produccion_t >= 0',engine="python").copy()
# Comprobación de equivalencia entre consulta_loc y consulta_query 
pd.testing.assert_frame_equal(consulta_loc, consulta_query)

#La ejecución completa es la verificación, por eso el display del mensaje
print("Ambos DataFrames (producto de consulta_loc y consulta_query) son idénticos.")
print(f"Filas en consulta_query: {len(consulta_query):,}")

Ambos DataFrames (producto de consulta_loc y consulta_query) son idénticos.
Filas en consulta_query: 4,924


In [5]:
#4. Conteos y porcentajes

# Agrupación por año y cultivo para cel conteo del número de registros
resumen = (
    consulta_loc.groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

# Total de registros por año y porcentaje
resumen["total_anio"] = resumen.groupby("anio")["registros"].transform("sum") #Registros en un año
resumen["porcentaje_anual"] = (resumen["registros"] / resumen["total_anio"]) * 100 #Porcentaje

# Ordenar por año (forma ascendente) y cantidad de registros (de forma descendente)
resumen = resumen.sort_values(by=["anio", "registros"], ascending=[True, False]).reset_index(drop=True)

# Verificación de conciliación de totales y suma de porcentajes
#(Esto es: suma_registros debe dar el n° de filas de consulta_loc y suma_porcentaje debe dar 100%)
print("--- Verificación de totales y porcentajes por año ---")
verificacion = resumen.groupby("anio").agg(
    suma_registros=("registros", "sum"),
    suma_porcentaje=("porcentaje_anual", "sum")
)
display(verificacion)

print("\n--- Verificación 2) Primeros 10 registros del resumen ---")
display(resumen.head(10))

--- Verificación de totales y porcentajes por año ---


,suma_registros,suma_porcentaje
anio,,
2023,2436,100.0
2024,2488,100.0



--- Verificación 2) Primeros 10 registros del resumen ---


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.811166
1,2023,Maíz,194,2436,7.963875
2,2023,Arveja,176,2436,7.224959
3,2023,Frijol,162,2436,6.650246
4,2023,Tomate,113,2436,4.638752
5,2023,Cebolla de bulbo,93,2436,3.817734
6,2023,Yuca,82,2436,3.366174
7,2023,Caña,63,2436,2.586207
8,2023,Trigo,58,2436,2.380952
9,2023,Mora,51,2436,2.093596


In [6]:
#5. Tabla dinámica y conclusiones.
# Construccion de la tabla dinámica (se plantea cultivos en filas y años en columnas)
tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    fill_value=0,
    aggfunc="sum"
)

# Display de primeras tuplas
print("--- Tabla dinámica de registros por cultivo y año ---")
display(tabla_dinamica.head(10))

# Exportación el DataFrame resumen a la ruta configurada en OUT (de la configuración inicial)
archivo_salida = OUT / "resumen_eva.csv"
resumen.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n¡No hay errores, resumen exportado a: {archivo_salida}!")


--- Tabla dinámica de registros por cultivo y año ---


anio,2023,2024
Cultivo,,
Acelga,5,8
Achira,4,4
Agraz - mortiño,3,4
Aguacate,48,52
Ahuyama,23,26
Ajo,7,7
Ají,1,2
Apio,0,1
Arazá,3,4



¡No hay errores, resumen exportado a: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv!


**Interpretación del resultado y conclusiones:**

Se estructuró la tabla dinámica para evaluar la presencia de cada cultivo por año y completando con 0 las combinaciones sin observaciones. El archivo `resumen_eva.csv` quedó guardado en la ruta de salidas OUT configurada.

**Conclusiones principales:**
1. **Incremento en la cobertura de reporte municipal:** El número total de registros agrícolas válidos evaluados en el departamento de Boyacá aumentó de 2.436 en el año 2023 a 2.488 en el año 2024 (un incremento del 2.13%). Esta tendencia se observa en la tabla dinámica para cultivos individuales como el Aguacate (paso de 48 a 52), la Ahuyama (paso de 23 a 26) y la Arracacha (de 48 a 51), y la aparición del Apio (de 0 a 1), reflejando una extensión en el monitoreo agropecuario.
2. **Masividad de cultivos tradicionales** A pesar de haber una alta diversidad en cultivos, la mayor concentración de evaluaciones y cobertura municipal en Boyacá corresponde a la Papa (con 239 registros en 2023, equivalentes al 9.81% del total) y al Maíz (con 194 registros, 7.96%), seguidos por la Arveja y el Frijol.